In [19]:
import mlflow
import pandas as pd
import pickle
import seaborn as sns
import os

from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

In [ ]:
#mlfow setup
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("homework")

#autologging for sklearn
mlflow.sklearn.autolog()

def load_pickle(filename: str):
    with open(filename, 'rb') as f_in:
        return pickle.load(f_in)
    
def run_train(data_path: str):
    
    X_train, y_train = load_pickle(os.path.join(data_path, 'train.pkl'))
    X_val, y_val = load_pickle(os.path.join(data_path, 'val.pkl'))

    with mlflow.start_run():
        rf = RandomForestRegressor(max_depth=10, random_state=0)
        rf.fit(X_train, y_train)

        y_pred = rf.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)

        print(f"RMSE: {rmse}")
    
data_path = "../output"

run_train(data_path)


2026/03/28 14:57:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RMSE: 5.431162180141208
🏃 View run selective-elk-74 at: http://127.0.0.1:5000/#/experiments/1/runs/5a6a0b1b771f40eeaeb5c3beea873b03
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [ ]:
def dump_pickle(obj, filename):
    with open(filename, 'wb') as f_out:
        pickle.dump(obj, f_out)

In [15]:
def read_dataframe(filename:str):
    df = pd.read_parquet(filename)
    
    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60


    df = df[(df['duration'] >= 1) & (df['duration'] <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    return df


In [16]:
def preprocess(df: pd.DataFrame, dv: DictVectorizer, fit_dv: bool = False):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']

    categorical = ['PU_DO']
    numerical = ['trip_distance']

    dicts = df[categorical + numerical].to_dict(orient='records')

    if fit_dv:
        X = dv.fit_transform(dicts)
    else:
        X = dv.transform(dicts)

    return X, dv

In [17]:
def run_preprocessing(raw_data_path: str, dest_path: str):
    os.makedirs(dest_path, exist_ok=True)
    
    # Load parquet files
    df_train = read_dataframe(os.path.join(raw_data_path, 'green_tripdata_2023-01.parquet'))
    df_val = read_dataframe(os.path.join(raw_data_path, 'green_tripdata_2023-02.parquet'))
    df_test = read_dataframe(os.path.join(raw_data_path, 'green_tripdata_2023-03.parquet'))

    # Extract target
    target = 'duration'
    y_train = df_train[target].values
    y_val = df_val[target].values
    y_test = df_test[target].values

    # Fit DictVectorizer on training data
    dv = DictVectorizer()
    X_train, dv = preprocess(df_train, dv, fit_dv=True)
    X_val, _ = preprocess(df_val, dv, fit_dv=False)
    X_test, _ = preprocess(df_test, dv, fit_dv=False)

    # Save DictVectorizer and datasets
    dump_pickle(dv, os.path.join(dest_path, "dv.pkl"))
    dump_pickle((X_train, y_train), os.path.join(dest_path, "train.pkl"))
    dump_pickle((X_val, y_val), os.path.join(dest_path, "val.pkl"))
    dump_pickle((X_test, y_test), os.path.join(dest_path, "test.pkl"))
    
    print(f"Files saved to {dest_path}:")
    print(os.listdir(dest_path))
    print(f"\nTotal files: {len(os.listdir(dest_path))}")

# Your specific paths
raw_data_path = "../data"
dest_path = "../output"

# Run preprocessing
run_preprocessing(raw_data_path, dest_path)

Files saved to ../output:
['dv.pkl', 'test.pkl', 'train.pkl', 'val.pkl']

Total files: 4
